# Sprint 8 · Webinar 25 · Data Analytics teórico (Explorar conexiones con correlaciones) · Student Version

**Programa:** Data Analytics  
**Sprint:** 8  
**Modalidad:** Teórica con demostraciones en Python  
**Duración sugerida:** 120 minutos  

Este cuaderno está diseñado para que **tomes apuntes durante la sesión**, ejecutes ejemplos guiados y dejes evidencia de tus interpretaciones.

## ¿Cómo usar esta versión?
- Completa definiciones, tablas y preguntas de reflexión.
- Ejecuta las celdas de código y registra lo que observas.
- En los bloques con `TODO`, modifica variables, gráficos o conclusiones.
- Usa este notebook como base de estudio para la sesión práctica.


## Fecha

Completa la información de la sesión:

- **Fecha:** `____ / ____ / 2026`
- **Instructor/a:** `____________________________`
- **Grupo / Cohorte:** `____________________________`
- **Tema central de la clase:** `Correlaciones, relaciones y comunicación responsable`


## Objetivos de la sesión teórica

Al finalizar esta sesión, deberías poder:

1. Explicar qué es **correlación** y diferenciarla de **causalidad**.
2. Explorar relaciones entre variables usando visualizaciones como `scatterplots`, `heatmaps` y `scatter matrix`.
3. Calcular e interpretar **Pearson** y **Spearman**.
4. Escoger una métrica razonable cuando las variables no son del mismo tipo.
5. Reconocer riesgos de interpretación: confusión, causalidad inversa y **Simpson’s paradox**.
6. Redactar hallazgos de correlación de forma clara, ética y accionable.

### Mis metas personales para esta sesión
- `___________________________________________________________`
- `___________________________________________________________`


## Agenda sugerida (120 minutos)

| Tiempo | Bloque | Contenido | Lo entendí / dudas |
|---:|---|---|---|
| 0–10 min | Calentamiento | Relación, correlación y causalidad | `__` |
| 10–30 min | Dataset | Creación y exploración rápida del dataset | `__` |
| 30–55 min | 8.1 | Relaciones visuales: líneas, scatterplots, heatmaps | `__` |
| 55–85 min | 8.2 | Cálculo de correlaciones y tipos de variables | `__` |
| 85–100 min | Riesgos | Confusores y Simpson’s paradox | `__` |
| 100–112 min | Automatización | Funciones para perfilar relaciones | `__` |
| 112–118 min | Comunicación | Ética, reportes y apoyo con IA | `__` |
| 118–120 min | Cierre | Takeaways y siguientes pasos | `__` |

### Nota rápida
Mientras avance la clase, marca cada bloque como: `claro / repasar / confuso`.


## Ejercicio 0 · Calentamiento en breakout rooms (discusión conceptual, 10 min)

**Objetivo:** activar intuición y vocabulario antes de entrar en código.

### Discusión en equipos
1. Den **2 ejemplos** de variables que podrían estar relacionadas en negocio o vida real.
   - `___________________________________________________________`
   - `___________________________________________________________`

2. Con sus palabras, ¿qué significa que una relación sea **lineal**?
   - `___________________________________________________________`

3. ¿Qué entienden por relación **monótona**?
   - `___________________________________________________________`

4. ¿Puede haber una correlación alta sin causalidad? Den un ejemplo.
   - `___________________________________________________________`

5. ¿Qué riesgo hay si mostramos un coeficiente sin contexto ni gráfico?
   - `___________________________________________________________`

### Idea que compartirá mi grupo
`___________________________________________________________`


## Ejercicio 1 · Crear y explorar un dataset extenso para análisis (20 min)

Vamos a trabajar con un dataset sintético que mezcla varios tipos de variables:

- **Numéricas:** gasto, revenue, tiempo en sitio, temperatura, etc.
- **Binarias:** `is_premium`, `converted`
- **Categóricas:** canal, región, dispositivo, segmento
- **Ordinales:** satisfacción de 1 a 5
- **Fecha:** semana

El dataset incluye dos situaciones didácticas importantes:
- **Confusión:** temperatura influye tanto en `ice_cream_sales` como en `drownings`.
- **Simpson’s paradox:** la relación entre `marketing_spend_usd` y `converted` cambia al segmentar.

### Antes de ejecutar
- ¿Qué variables crees que tendrán una relación positiva?  
  `___________________________________________________________`
- ¿Qué variables crees que podrían mostrar una relación engañosa?  
  `___________________________________________________________`
- ¿Qué gráfico usarías primero para revisar una relación entre dos variables numéricas?  
  `___________________________________________________________`


In [ ]:
# 1) Setup del entorno
# Ejecuta esta celda primero.
# Si aparece un error de librerías, anota aquí cuál fue y cómo lo resolviste.

# ============================================================
# Imports y configuración (ejecuta esta celda primero)
# ============================================================

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

RANDOM_SEED = 42


In [ ]:
# 2) Generación del dataset sintético
# Lee los comentarios de la función y ubica:
# - variables numéricas
# - variables categóricas
# - el caso de confusión
# - el caso de Simpson's paradox
#
# Nota de apuntes:
# ¿Qué ventaja tiene usar un dataset sintético para aprender este tema?

# ============================================================
# Generación del dataset sintético para el Sprint 8
# ============================================================

def generate_sprint8_dataset(n_rows: int = 8000, seed: int = 42) -> pd.DataFrame:
    """
    Genera un dataset sintético para análisis de correlación y relaciones.

    Metas didácticas:
    - Incluir múltiples tipos de variables (numéricas, binarias, categóricas, ordinales, fecha).
    - Incluir un escenario de confusión:
        temperature -> ice_cream_sales y temperature -> drownings
    - Incluir un escenario de Simpson's paradox:
        marketing_spend_usd vs converted, al segmentar por 'segment'

    Parámetros
    ----------
    n_rows : int
        Número de filas.
    seed : int
        Semilla para reproducibilidad.

    Retorna
    -------
    pd.DataFrame
        Dataset sintético.
    """
    rng = np.random.default_rng(seed)

    # Tiempo (semanal)
    start = pd.Timestamp("2025-01-06")  # lunes
    weeks = pd.date_range(start, periods=104, freq="W-MON")  # ~2 años
    week = rng.choice(weeks, size=n_rows, replace=True)
    week_of_year = pd.to_datetime(week).isocalendar().week.astype(int).to_numpy()

    # Contexto categórico
    region = rng.choice(["Norte", "Centro", "Sur", "Oriente", "Occidente"], size=n_rows, p=[0.18, 0.28, 0.20, 0.17, 0.17])
    channel = rng.choice(["Orgánico", "Paid Search", "Social", "Email", "Referidos"], size=n_rows, p=[0.30, 0.28, 0.18, 0.14, 0.10])
    device = rng.choice(["Mobile", "Desktop", "Tablet"], size=n_rows, p=[0.62, 0.33, 0.05])

    # Clientes (repetidos)
    customer_id = rng.integers(1, 2001, size=n_rows)
    age = np.clip(rng.normal(34, 11, size=n_rows).round().astype(int), 18, 70)
    income = np.clip(rng.lognormal(mean=10.5, sigma=0.45, size=n_rows), 12000, 220000)  # ingreso anual
    is_premium = rng.choice([0, 1], size=n_rows, p=[0.78, 0.22])

    # Confusión: estacionalidad -> temperatura -> (helados, drownings)
    temperature_c = 24 + 6 * np.sin(2 * np.pi * (week_of_year / 52.0)) + rng.normal(0, 1.8, size=n_rows)
    ice_cream_sales = np.clip(200 + 16 * temperature_c + rng.normal(0, 40, size=n_rows), 0, None)
    drownings = np.clip(3 + 0.18 * temperature_c + rng.normal(0, 1.2, size=n_rows), 0, None)

    # Segmento (para Simpson)
    segment = rng.choice(["SMB", "Enterprise"], size=n_rows, p=[0.70, 0.30])

    # Marketing spend (segmento + canal)
    channel_uplift = pd.Series(channel).map({"Orgánico": 0.7, "Paid Search": 1.4, "Social": 1.1, "Email": 0.9, "Referidos": 0.6}).to_numpy()
    base_spend = np.where(segment == "Enterprise", 320, 90)
    marketing_spend_usd = np.clip(base_spend * channel_uplift + rng.normal(0, 35, size=n_rows), 0, None)

    # Engagement (relación no lineal con spend)
    sessions = np.clip((0.9 * np.sqrt(marketing_spend_usd) + rng.normal(0, 3, size=n_rows)), 0, None).round().astype(int)
    pages_viewed = np.clip((sessions * rng.normal(3.2, 0.6, size=n_rows) + rng.normal(0, 4, size=n_rows)), 1, None).round().astype(int)
    time_on_site_sec = np.clip((pages_viewed * rng.normal(26, 7, size=n_rows) + rng.normal(0, 60, size=n_rows)), 20, None)

    # Descuentos
    discount_pct = np.clip(rng.normal(12, 8, size=n_rows), 0, 40)

    # Conversión (probabilidad logística)
    seg_mean_spend = np.where(segment == "Enterprise", 320, 90)
    spend_centered = marketing_spend_usd - seg_mean_spend

    logit = (
        -2.2
        + 0.35 * is_premium
        + 0.05 * np.log1p(pages_viewed)
        + 0.03 * discount_pct
        + np.where(segment == "Enterprise", 0.8, -0.2)
        - 0.0032 * spend_centered  # dentro de segmento: leve relación negativa con spend
        + rng.normal(0, 0.35, size=n_rows)
    )

    conv_prob = 1 / (1 + np.exp(-logit))
    converted = (rng.random(n_rows) < conv_prob).astype(int)

    # Orders y revenue
    orders = np.where(converted == 1, np.clip(rng.poisson(1.15, size=n_rows), 1, 5), 0)
    base_aov = np.where(is_premium == 1, rng.normal(150, 40, size=n_rows), rng.normal(95, 28, size=n_rows))
    avg_order_value_usd = np.clip(base_aov * (1 - discount_pct / 120) + rng.normal(0, 10, size=n_rows), 15, None)
    revenue_usd = orders * avg_order_value_usd

    # Experiencia: delivery y tickets
    delivery_time_days = np.clip(rng.normal(3.2, 1.4, size=n_rows) + np.where(region == "Oriente", 0.8, 0.0), 0.5, 12)
    support_tickets = np.clip(rng.poisson(0.35 + 0.25 * (delivery_time_days > 5), size=n_rows), 0, None)

    # Satisfacción (ordinal 1–5)
    sat_latent = (
        3.8
        + 0.5 * is_premium
        - 0.22 * delivery_time_days
        - 0.35 * support_tickets
        + 0.08 * np.log1p(revenue_usd)
        + rng.normal(0, 0.55, size=n_rows)
    )
    satisfaction_1_5 = np.clip(np.round(sat_latent), 1, 5).astype(int)

    # Churn risk (0–1)
    churn_logit = (
        0.8
        - 0.75 * (satisfaction_1_5 - 3)
        + 0.28 * support_tickets
        + 0.10 * (delivery_time_days - 3)
        + rng.normal(0, 0.35, size=n_rows)
    )
    churn_risk = 1 / (1 + np.exp(-churn_logit))

    df = pd.DataFrame({
        "week": pd.to_datetime(week),
        "customer_id": customer_id,
        "region": region,
        "segment": segment,
        "channel": channel,
        "device": device,
        "age": age,
        "income_usd": np.round(income, 2),
        "is_premium": is_premium,
        "temperature_c": np.round(temperature_c, 2),
        "ice_cream_sales": np.round(ice_cream_sales, 0).astype(int),
        "drownings": np.round(drownings, 2),
        "marketing_spend_usd": np.round(marketing_spend_usd, 2),
        "sessions": sessions,
        "pages_viewed": pages_viewed,
        "time_on_site_sec": np.round(time_on_site_sec, 0).astype(int),
        "discount_pct": np.round(discount_pct, 2),
        "converted": converted,
        "orders": orders,
        "avg_order_value_usd": np.round(avg_order_value_usd, 2),
        "revenue_usd": np.round(revenue_usd, 2),
        "delivery_time_days": np.round(delivery_time_days, 2),
        "support_tickets": support_tickets,
        "satisfaction_1_5": satisfaction_1_5,
        "churn_risk": np.round(churn_risk, 4),
    })

    return df.sort_values(["week", "customer_id"]).reset_index(drop=True)

df = generate_sprint8_dataset(n_rows=8000, seed=RANDOM_SEED)
df.head()


In [ ]:
# 3) Guardar el dataset a CSV
# Esta parte es opcional. Si quieres conservar el archivo para otra sesión,
# descomenta la línea de guardado.

# Guardar el dataset a CSV (para usarlo en ejercicios o compartirlo)

csv_path = "sprint8_webinar23_correlations_dataset.csv"
#df.to_csv(csv_path, index=False)

print("CSV guardado en:", csv_path)


In [ ]:
# 4) Exploración rápida del dataset
# Después de ejecutar, registra:
# - forma del DataFrame
# - primeras impresiones sobre las columnas
# - cualquier dato que te llame la atención

# Exploración rápida (EDA mínima)

print("Forma (filas, columnas):", df.shape)
display(df.sample(5, random_state=RANDOM_SEED))



In [ ]:
# 5) Tipos de datos
# Úsalo para clasificar variables en numéricas, categóricas, binarias u ordinales.

print("\nTipos de datos:")
display(df.dtypes)

In [ ]:
# 6) Resumen de variables numéricas
# Observa rangos, medias y posibles valores extremos.

print("\nResumen de variables numéricas:")
display(df.select_dtypes(include="number").describe())

In [ ]:
# 7) Valores faltantes
# Si detectaras nulos relevantes, ¿los imputarías, eliminarías o documentarías?

print("\nValores faltantes (por columna):")
display(df.isna().sum().sort_values(ascending=False).head(10))

### Registro de hallazgos del Ejercicio 1

Completa estas notas después de ejecutar la exploración inicial:

- **Número aproximado de filas:** `________________`
- **Número aproximado de columnas:** `________________`
- **Variables numéricas que identifiqué:**  
  `___________________________________________________________`
- **Variables categóricas que identifiqué:**  
  `___________________________________________________________`
- **Variables ordinales o binarias que identifiqué:**  
  `___________________________________________________________`

### Hipótesis iniciales
1. Variable A: `________________` con Variable B: `________________`  
   Tipo de relación esperada: `positiva / negativa / incierta`

2. Variable A: `________________` con Variable B: `________________`  
   Tipo de relación esperada: `positiva / negativa / incierta`

### Observaciones rápidas
- Posibles nulos o problemas de calidad: `__________________________`
- Variable que me gustaría analizar primero: `__________________________`


## 8.1 Exploring Relationships Visually

Antes de calcular correlaciones, conviene **mirar** los datos.

### Idea clave
Un solo coeficiente puede esconder:
- **no linealidad**
- **outliers**
- **subgrupos**
- **cambios de comportamiento por segmento**

### Mientras avances
Completa:
- El gráfico que mejor muestra una relación temporal es: `________________`
- El gráfico que mejor muestra una relación entre dos variables numéricas es: `________________`
- El gráfico que resume muchas relaciones numéricas a la vez es: `________________`


### 8.1.1 Understanding Lineal Correlation & Relationships

Completa con tus palabras:

- **Correlación lineal:**  
  `___________________________________________________________`

- **Dirección positiva:**  
  `___________________________________________________________`

- **Dirección negativa:**  
  `___________________________________________________________`

- **Relación monótona:**  
  `___________________________________________________________`

> Pista: Pearson se enfoca en linealidad; Spearman suele capturar mejor relaciones monotónicas.


In [ ]:
# Ejemplo visual 1: relación en el tiempo
# Objetivo: comparar la evolución promedio semanal de algunas métricas.
# Antes de correr la gráfica, intenta predecir si spend y revenue se moverán parecido.

# Ejemplo visual 1: relación en el tiempo (líneas)
# Vamos a agregar por semana para visualizar tendencia.

weekly = (
    df.groupby("week", as_index=False)
      .agg(
          marketing_spend_usd=("marketing_spend_usd", "mean"),
          revenue_usd=("revenue_usd", "mean"),
          sessions=("sessions", "mean"),
          temperature_c=("temperature_c", "mean")
      )
)

display(weekly.head())

In [ ]:
# Interpreta la gráfica:
# - ¿ves patrones estacionales?
# - ¿spend y revenue se mueven juntos todo el tiempo?
# - ¿qué dudas te deja este gráfico?

plt.figure(figsize=(10, 4))
plt.plot(weekly["week"], weekly["marketing_spend_usd"], label="Marketing spend (prom.)")
plt.plot(weekly["week"], weekly["revenue_usd"], label="Revenue (prom.)")
plt.title("Relación temporal: gasto en marketing vs revenue")
plt.xlabel("Semana")
plt.ylabel("Valor promedio")
plt.legend()
plt.tight_layout()
plt.show()

### 8.1.2 Scatterplots for Relationships

El **scatterplot** es la visualización básica para estudiar relaciones entre variables numéricas.

### Qué debo buscar al mirar un scatterplot
- Tendencia general: `positiva / negativa / nula`
- Forma: `lineal / curva / dispersa`
- Presencia de outliers: `sí / no`
- Posibles subgrupos: `sí / no`

### Antes de ejecutar
Predice qué esperas ver en:
- `marketing_spend_usd` vs `revenue_usd`: `________________________`
- `marketing_spend_usd` vs `converted`: `________________________`


In [ ]:
# Scatterplot 1
# Observa forma, dispersión y posibles outliers.

# Scatterplot 1: spend vs revenue
plt.figure(figsize=(6, 5))
sns.scatterplot(data=df, x="marketing_spend_usd", y="revenue_usd", alpha=0.35)
plt.title("Scatterplot: marketing_spend_usd vs revenue_usd")
plt.tight_layout()
plt.show()

In [ ]:
# Scatterplot 2
# Aquí se usa color por segmento para revelar posibles subgrupos.
# Pregunta clave: ¿la relación total cambia cuando miras cada segmento?

# Scatterplot 2: colorear por segmento (para ver subgrupos)
plt.figure(figsize=(6, 5))
sns.scatterplot(data=df, x="marketing_spend_usd", y="converted", hue="segment", alpha=0.35)
plt.title("Spend vs converted (hue = segment)")
plt.tight_layout()
plt.show()


### 8.1.3 Multi-Variable Relationships: Heatmaps & Scatter Matrix

Cuando hay varias variables numéricas, necesitamos herramientas que permitan ver el panorama general.

### Completa la idea
- **Heatmap:** sirve para `________________________________________`
- **Scatter matrix:** sirve para `__________________________________`

### Checklist durante la demo
- [ ] Entendí por qué no conviene usar demasiadas columnas a la vez
- [ ] Identifiqué al menos una correlación alta o moderada
- [ ] Identifiqué al menos un par que necesita validación visual adicional


In [ ]:
# Selección de columnas numéricas para el análisis multivariable
# Puedes agregar o quitar columnas si quieres probar otra combinación.

# Selecciona un subconjunto razonable de variables numéricas
num_cols = [
    "marketing_spend_usd", "sessions", "pages_viewed", "time_on_site_sec",
    "discount_pct", "orders", "avg_order_value_usd", "revenue_usd",
    "delivery_time_days", "support_tickets", "satisfaction_1_5", "churn_risk",
    "temperature_c", "ice_cream_sales", "drownings"
]

corr = df[num_cols].corr(numeric_only=True)

In [ ]:
# Tabla de correlaciones del subconjunto numérico
# Revisa qué pares parecen fuertes, moderados o débiles.
# No saques conclusiones definitivas sin revisar gráficos.

df[num_cols].corr(numeric_only=True)

In [ ]:
# Heatmap de correlaciones
# Marca 2 pares de variables que te llamen la atención y luego justifica por qué.

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=False, cmap="coolwarm", center=0)
plt.title("Heatmap de correlaciones (subconjunto numérico)")
plt.tight_layout()
plt.show()


In [ ]:
# Scatter matrix
# Puede tardar un poco. Aquí se toma una muestra para que sea más manejable.

# Scatter matrix (puede tardar un poco). Usamos un sample para que sea más rápido.
from pandas.plotting import scatter_matrix

sample = df[num_cols].sample(600, random_state=RANDOM_SEED)

In [ ]:
# Después de ejecutar:
# - elige un par que merezca más análisis
# - anota si ves linealidad, curvatura u outliers

axes = scatter_matrix(sample, figsize=(14, 14), diagonal="kde", alpha=0.4)
plt.suptitle("Scatter matrix (muestra de 600 filas)", y=1.02)
plt.show()

### Mini-ejercicio (5 min)

Usa una celda nueva o modifica una existente para responder:

1. Elige **2 variables numéricas** y crea un scatterplot.
2. Describe si ves relación **lineal**, **monótona** o **ninguna**.
3. Anota si detectas **outliers** y cómo podrían afectar la correlación.

### Registro
- Variables elegidas: `________________` y `________________`
- Tipo de relación observada: `___________________________________`
- Riesgo de interpretación: `______________________________________`


## 8.2 Calculating Correlations

Ahora pasamos de la intuición visual a la cuantificación.

### Idea clave para anotar
Un coeficiente de correlación es un **resumen**, no una explicación completa.

### Antes de calcular
Responde:
- ¿Qué ventaja tiene calcular un coeficiente después de ver el gráfico?  
  `___________________________________________________________`
- ¿Qué problema puede aparecer si interpreto un coeficiente sin segmentar?  
  `___________________________________________________________`


### 8.2.1 Pearson and Spearman Correlations

Completa la siguiente tabla durante la explicación:

| Métrica | ¿Qué mide mejor? | ¿Cuándo usarla? | Riesgo o limitación |
|---|---|---|---|
| Pearson | `__________` | `__________` | `__________` |
| Spearman | `__________` | `__________` | `__________` |

### Interpretación rápida
- Si `r ≈ 1`, entonces: `________________________________________`
- Si `r ≈ -1`, entonces: `_______________________________________`
- Si `r ≈ 0`, entonces: `________________________________________`


In [ ]:
# Pearson vs Spearman para un par de variables
# Compara ambos coeficientes y anota si cuentan una historia parecida o distinta.

# Pearson vs Spearman para un par de variables

x = df["pages_viewed"]
y = df["revenue_usd"]

pearson_r, pearson_p = stats.pearsonr(x, y)
spearman_r, spearman_p = stats.spearmanr(x, y)

print("pages_viewed vs revenue_usd")
print(f"Pearson  r = {pearson_r:.3f} | p-value = {pearson_p:.3e}")
print(f"Spearman r = {spearman_r:.3f} | p-value = {spearman_p:.3e}")

# TODO opcional:
# prueba otro par de variables y compara si Pearson y Spearman cambian.

In [ ]:
# Visual de apoyo para interpretar los coeficientes del par anterior.

# Visual simple
plt.figure(figsize=(6, 5))
sns.scatterplot(data=df.sample(1200, random_state=RANDOM_SEED), x="pages_viewed", y="revenue_usd", alpha=0.35)
plt.title("pages_viewed vs revenue_usd (muestra)")
plt.tight_layout()
plt.show()

# TODO opcional:
# cambia las variables del gráfico si quieres validar otro par.

### 8.2.2 Correlación entre distintos tipos de variables

No todas las relaciones se resuelven con Pearson.

### Regla práctica
Primero identifica:
1. Tipo de variable A
2. Tipo de variable B
3. Pregunta analítica: linealidad, monotonicidad o dependencia general

### Completa este cuadro
| Tipo A | Tipo B | Métrica razonable |
|---|---|---|
| Numérica | Numérica | `________________` |
| Numérica | Binaria | `________________` |
| Categórica | Numérica | `________________` |
| Categórica | Categórica | `________________` |
| Ordinal | Numérica / ordinal | `________________` |

> En esta clase buscamos escoger una métrica **razonable** y explicar por qué.


In [ ]:
# Funciones auxiliares
# Lee la lógica general, no necesitas memorizar cada línea.
# Concéntrate en entender:
# - qué tipo de par resuelve cada función
# - qué valor devuelve

# Funciones sencillas para correlación según tipos de variables

def cramers_v(x: pd.Series, y: pd.Series) -> float:
    """Cramér's V para dos variables categóricas."""
    confusion = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(confusion)[0]
    n = confusion.sum().sum()
    r, k = confusion.shape

    # Corrección de sesgo (opcional pero útil en tablas pequeñas)
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)

    return np.sqrt(phi2corr / max(1e-12, min((kcorr-1), (rcorr-1))))

def correlation_ratio(categories: pd.Series, values: pd.Series) -> float:
    """Correlation ratio (η) para categórica -> numérica."""
    df_tmp = pd.DataFrame({"cat": categories, "val": values}).dropna()
    if df_tmp["cat"].nunique() < 2:
        return np.nan

    grand_mean = df_tmp["val"].mean()
    ss_between = df_tmp.groupby("cat")["val"].count().mul((df_tmp.groupby("cat")["val"].mean() - grand_mean) ** 2).sum()
    ss_total = ((df_tmp["val"] - grand_mean) ** 2).sum()

    return np.sqrt(ss_between / ss_total) if ss_total > 0 else np.nan


In [ ]:
# Ejemplo 1: numérica vs binaria
# Interpreta si una variable binaria parece asociarse con una numérica.

# Ejemplos:
# 1) Numérica vs Binaria: point-biserial
r_pb, p_pb = stats.pointbiserialr(df["converted"], df["revenue_usd"])
print(f"Point-biserial (converted vs revenue_usd): r={r_pb:.3f}, p={p_pb:.3e}")


# TODO opcional:
# cambia las variables y prueba otro ejemplo numérica vs binaria.

In [ ]:
# Ejemplo 2: categórica (k>2) vs numérica
# Piensa este resultado como tamaño de efecto, no como causalidad.

# 2) Categórica (k>2) vs Numérica: correlation ratio (η)
eta = correlation_ratio(df["channel"], df["revenue_usd"])
print(f"Correlation ratio η (channel -> revenue_usd): eta={eta:.3f}")

# TODO opcional:
# reemplaza 'channel' o 'revenue_usd' por otra combinación categórica-numérica.

In [ ]:
# Ejemplo 3: categórica vs categórica
# Útil para ver dependencia entre categorías.

# 3) Categórica vs Categórica: Cramér's V
v = cramers_v(df["device"], df["channel"])
print(f"Cramér's V (device vs channel): V={v:.3f}")

# TODO opcional:
# prueba otra pareja de variables categóricas.

In [ ]:
# Ejemplo 4: ordinal vs numérica
# Aquí tratamos la variable ordinal como rangos.

# 4) Ordinal vs Numérica: Spearman (tratando ordinal como ranks)
rho, p = stats.spearmanr(df["satisfaction_1_5"], df["churn_risk"])
print(f"Spearman (satisfaction_1_5 vs churn_risk): rho={rho:.3f}, p={p:.3e}")

# TODO opcional:
# busca otra variable ordinal o construye un ranking simple.

### 8.2.3 Avoiding Misinterpretations (Correlation not equal Causation)

**Correlación ≠ causalidad.**

### Tipos de error frecuentes
- **Confusor:** `_________________________________________________`
- **Causalidad inversa:** `_________________________________________`
- **Coincidencia o relación espuria:** `____________________________`
- **Sesgo de selección:** `_________________________________________`

### Nota de clase
En este dataset, `temperature_c` funciona como posible variable confusora para explicar la relación entre:
- `_____________________` y `_____________________`


In [ ]:
# Caso de confusión
# El objetivo no es solo ver una correlación alta, sino preguntarnos qué tercera variable podría explicarla.

# Correlación espuria por confusión: helados vs drownings

r_xy, _ = stats.pearsonr(df["ice_cream_sales"], df["drownings"])
r_xt, _ = stats.pearsonr(df["ice_cream_sales"], df["temperature_c"])
r_yt, _ = stats.pearsonr(df["drownings"], df["temperature_c"])

print(f"Pearson(ice_cream_sales, drownings)   = {r_xy:.3f}")
print(f"Pearson(ice_cream_sales, temperature)= {r_xt:.3f}")
print(f"Pearson(drownings, temperature)      = {r_yt:.3f}")

In [ ]:
# Gráfico del posible confusor
# Observa cómo el color ayuda a revelar la variable de contexto.

# Visualizar el patrón
sample = df.sample(1500, random_state=RANDOM_SEED)

plt.figure(figsize=(6, 5))
sns.scatterplot(data=sample, x="ice_cream_sales", y="drownings", hue="temperature_c", palette="viridis", alpha=0.6)
plt.title("Confusión: ice_cream_sales vs drownings (color = temperature_c)")
plt.tight_layout()
plt.show()

In [ ]:
# Estratificación por bins de temperatura
# Compara la relación global con la relación dentro de grupos.

# Estratificar por 'bins' de temperatura para ver que la relación se debilita dentro de grupos
bins = pd.cut(df["temperature_c"], bins=5)
within = df.groupby(bins).apply(lambda g: stats.pearsonr(g["ice_cream_sales"], g["drownings"])[0]).rename("pearson_within_bin")
display(within)

### 8.2.4 Understanding Simpson's Paradox

**Simpson’s paradox** ocurre cuando una relación cambia al separar los datos por un subgrupo relevante.

### Completa con tus palabras
- Relación observada en el total: `________________________________`
- Subgrupo que puede cambiar la lectura: `___________________________`
- Riesgo para negocio si no segmentamos: `___________________________`

### Pregunta guía
¿Por qué un promedio general puede ocultar comportamientos distintos entre segmentos?
`___________________________________________________________`


In [ ]:
# Caso de Simpson's paradox
# Primero miramos la relación agregada; después, por segmento.

# Simpson's paradox: spend vs conversion (global vs por segmento)

# Convertimos converted (0/1) en "tasa" por bins de spend para visualizar
df_tmp = df.copy()
df_tmp["spend_bin"] = pd.qcut(df_tmp["marketing_spend_usd"], q=10, duplicates="drop")

overall = df_tmp.groupby("spend_bin")["converted"].mean()
by_segment = df_tmp.groupby(["segment", "spend_bin"])["converted"].mean().reset_index()

display(overall.head())
display(by_segment.head())

In [ ]:
# Gráfico agregado
# Toma nota de la tendencia que parece verse en el total.

plt.figure(figsize=(10, 4))
plt.plot(overall.index.astype(str), overall.values, marker="o", label="Total")
plt.xticks(rotation=45, ha="right")
plt.title("Tasa de conversión por bins de marketing_spend_usd (Total)")
plt.ylabel("mean(converted)")
plt.tight_layout()
plt.show()


In [ ]:
# Compara correlación total vs por segmento
# Si cambian de signo o magnitud, eso merece explicación.

# Correlación simple (Pearson) total vs por segmento
r_total, _ = stats.pearsonr(df["marketing_spend_usd"], df["converted"])
r_smb, _ = stats.pearsonr(df[df["segment"]=="SMB"]["marketing_spend_usd"], df[df["segment"]=="SMB"]["converted"])
r_ent, _ = stats.pearsonr(df[df["segment"]=="Enterprise"]["marketing_spend_usd"], df[df["segment"]=="Enterprise"]["converted"])

print(f"Pearson(total):      {r_total:.3f}")
print(f"Pearson(SMB):        {r_smb:.3f}")
print(f"Pearson(Enterprise): {r_ent:.3f}")

### 8.2.5 Automating Correlation Analysis with Python Functions

En un caso real, no siempre calculamos relaciones una por una.

### Ventajas de automatizar
- `___________________________________________________________`
- `___________________________________________________________`
- `___________________________________________________________`

### Mientras revisas la función
Marca lo que identifiques:
- [ ] Entendí cómo infiere el tipo de variable
- [ ] Vi cómo elige una métrica según el par de variables
- [ ] Entendí cómo ordena los resultados
- [ ] Detecté una limitación o simplificación de la función


In [ ]:
# Automatización del análisis
# Esta función genera una tabla resumen de relaciones.
# Mientras la revisas, intenta identificar qué simplificaciones usa.

# Automatizar: un profiler básico de correlaciones según tipos

def infer_var_type(s: pd.Series) -> str:
    """Clasifica el tipo de variable: numeric / binary / categorical / ordinal."""
    if pd.api.types.is_numeric_dtype(s):
        # Heurística: 0/1 -> binary; 1–5 -> ordinal (si pocos valores enteros)
        uniques = pd.Series(s.dropna().unique())
        if len(uniques) <= 2 and set(uniques.astype(float)) <= {0.0, 1.0}:
            return "binary"
        if len(uniques) <= 7 and pd.api.types.is_integer_dtype(s):
            return "ordinal"
        return "numeric"
    else:
        return "categorical"

def correlation_pair(a: pd.Series, b: pd.Series) -> dict:
    """
    Calcula una métrica razonable de relación para el par (a, b), según tipos.

    Retorna un dict con:
    - metric
    - value
    - p_value (cuando aplica)
    - type_a, type_b
    """
    type_a = infer_var_type(a)
    type_b = infer_var_type(b)

    # Acomodar para usar reglas simétricas
    # (tratamos algunos casos direccionados, p.ej. eta categórica->numérica)
    if type_a == "numeric" and type_b == "numeric":
        r, p = stats.pearsonr(a, b)
        return {"metric": "pearson", "value": r, "p_value": p, "type_a": type_a, "type_b": type_b}

    if type_a == "ordinal" and type_b in ["ordinal", "numeric"]:
        r, p = stats.spearmanr(a, b)
        return {"metric": "spearman", "value": r, "p_value": p, "type_a": type_a, "type_b": type_b}

    if type_b == "ordinal" and type_a == "numeric":
        r, p = stats.spearmanr(a, b)
        return {"metric": "spearman", "value": r, "p_value": p, "type_a": type_a, "type_b": type_b}

    if type_a == "binary" and type_b == "numeric":
        r, p = stats.pointbiserialr(a, b)
        return {"metric": "point_biserial", "value": r, "p_value": p, "type_a": type_a, "type_b": type_b}

    if type_b == "binary" and type_a == "numeric":
        r, p = stats.pointbiserialr(b, a)
        return {"metric": "point_biserial", "value": r, "p_value": p, "type_a": type_a, "type_b": type_b}

    if type_a == "categorical" and type_b == "categorical":
        v = cramers_v(a, b)
        return {"metric": "cramers_v", "value": v, "p_value": np.nan, "type_a": type_a, "type_b": type_b}

    if type_a == "categorical" and type_b == "numeric":
        eta = correlation_ratio(a, b)
        return {"metric": "eta", "value": eta, "p_value": np.nan, "type_a": type_a, "type_b": type_b}

    if type_b == "categorical" and type_a == "numeric":
        eta = correlation_ratio(b, a)
        return {"metric": "eta", "value": eta, "p_value": np.nan, "type_a": type_a, "type_b": type_b}

    # Fallback: Spearman (por si no encaja exactamente)
    r, p = stats.spearmanr(pd.factorize(a)[0], pd.factorize(b)[0])
    return {"metric": "spearman_factorized_fallback", "value": r, "p_value": p, "type_a": type_a, "type_b": type_b}

def correlation_profile(df: pd.DataFrame, cols: list[str], top_n: int = 15) -> pd.DataFrame:
    """Calcula métricas para todos los pares de columnas en cols."""
    results = []
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            c1, c2 = cols[i], cols[j]
            a = df[c1].dropna()
            b = df[c2].dropna()
            # Alinear índices
            aligned = pd.concat([a, b], axis=1).dropna()
            if aligned.shape[0] < 30:
                continue
            res = correlation_pair(aligned.iloc[:, 0], aligned.iloc[:, 1])
            res.update({"var_1": c1, "var_2": c2, "n": aligned.shape[0]})
            results.append(res)

    out = pd.DataFrame(results)
    # Ordenar por "fuerza" (abs(value)) para métricas tipo correlación; en eta/cramers también usamos value
    out["abs_value"] = out["value"].abs()
    return out.sort_values("abs_value", ascending=False).head(top_n).drop(columns=["abs_value"])

cols_for_profile = [
    "marketing_spend_usd", "sessions", "pages_viewed", "time_on_site_sec",
    "discount_pct", "orders", "avg_order_value_usd", "revenue_usd",
    "delivery_time_days", "support_tickets", "satisfaction_1_5", "churn_risk",
    "segment", "channel", "device", "converted"
]

top = correlation_profile(df, cols_for_profile, top_n=20)
display(top)


### Mini-ejercicio (5 min)

Elige una variable objetivo, por ejemplo `revenue_usd` o `churn_risk`, y analiza 3–5 variables candidatas.

### Evidencia que debes registrar
- Variable objetivo: `____________________________`
- Variables candidatas: `___________________________________________`
- Métricas usadas: `_______________________________________________`

### Conclusiones tentativas
1. `___________________________________________________________`
2. `___________________________________________________________`

### Advertencia metodológica
- Posible confusor, subgrupo o limitación detectada:  
  `___________________________________________________________`


## 8.3 Communicating Correlation Insights

El análisis no termina al obtener números. Termina cuando una persona del negocio entiende:

- qué relación encontramos,
- qué significa,
- qué no podemos afirmar,
- y qué haríamos después.

### Mi audiencia objetivo
- ¿A quién le contaría este análisis? `____________________________`
- ¿Qué decisión de negocio podría apoyar? `________________________`


### 8.3.1 Ethics of Correlation Analysis & Responsible Data Reporting

Completa este checklist de comunicación responsable:

- [ ] No presenté correlación como causalidad
- [ ] Expliqué la muestra o población analizada
- [ ] Mencioné filtros, supuestos o limitaciones
- [ ] Consideré sesgos o variables sensibles
- [ ] Puedo reproducir el análisis con mi notebook

### Reflexión breve
¿Qué puede salir mal si un stakeholder interpreta una correlación como causalidad?
`___________________________________________________________`


### 8.3.2 Writing a Correlation Report

Usa esta estructura corta para tus apuntes de reporte:

1. **Contexto:** `_______________________________________________`
2. **Datos:** `___________________________________________________`
3. **Método:** `_________________________________________________`
4. **Hallazgos:** `______________________________________________`
5. **Limitaciones:** `___________________________________________`
6. **Próximos pasos:** `__________________________________________`

### Mi frase inicial del reporte
`___________________________________________________________`


In [ ]:
# Borrador automático de reporte
# Úsalo como punto de partida, no como versión final.
# Tu tarea como analista es revisar, editar y contextualizar el texto.

# Mini-reporte automatizado (borrador) en texto

def format_corr_row(row: pd.Series) -> str:
    metric = row["metric"]
    v = row["value"]
    p = row["p_value"]
    p_txt = f", p={p:.2e}" if pd.notna(p) else ""
    return f"- {row['var_1']} ↔ {row['var_2']}: {metric}={v:.3f}{p_txt} (n={int(row['n'])})"

top10 = correlation_profile(df, cols_for_profile, top_n=10)

print("CORRELATION REPORT (BORRADOR)\n")
print("1) Contexto")
print("   Explorar relaciones entre variables de experiencia, marketing y revenue para generar hipótesis.\n")

print("2) Datos")
print(f"   Dataset sintético: {df.shape[0]} filas, {df.shape[1]} columnas; granularidad semanal.\n")

print("3) Método")
print("   Métricas según tipos: pearson/spearman/point-biserial/eta/cramer's V.\n")

print("4) Hallazgos principales (top 10 por magnitud)")
for _, r in top10.iterrows():
    print(format_corr_row(r))

print("\n5) Limitaciones")
print("   - Las correlaciones no implican causalidad.")
print("   - Puede existir confusión (p.ej., temperatura) y Simpson's paradox (p.ej., segmento).")
print("   - Dataset sintético: valores y efectos fueron simulados.\n")

print("6) Próximos pasos")
print("   - Validar hipótesis con segmentación adicional.")
print("   - Diseñar experimentos (A/B) o análisis causal cuando aplique.")


# TODO opcional:
# adapta el reporte para una audiencia específica: marketing, producto o dirección.

### 8.3.3 Correlation Reports with AI

La IA puede apoyar la redacción, pero no reemplaza el criterio analítico.

### Buenas prácticas
- No compartir datos sensibles sin anonimización
- Entregar contexto, supuestos y limitaciones
- Pedir explícitamente que marque incertidumbre
- Verificar que el resumen no invente conclusiones

### Prompt que quiero adaptar
> “Tengo una tabla con correlaciones y algunos gráficos.  
> Resume 5 hallazgos, explica limitaciones, sugiere posibles confusores y próximos pasos.  
> No inventes datos; si falta información, dilo explícitamente.”

### Qué parte del prompt modificaría para mi caso
`___________________________________________________________`


## 6. Takeaways de la sesión teórica

Completa las ideas clave con tus propias palabras:

1. Visualizar antes de calcular es importante porque  
   `___________________________________________________________`

2. Pearson y Spearman se diferencian en  
   `___________________________________________________________`

3. El tipo de variable importa porque  
   `___________________________________________________________`

4. Correlación no implica causalidad porque  
   `___________________________________________________________`

5. Simpson’s paradox me recuerda que  
   `___________________________________________________________`

6. Comunicar bien un análisis implica  
   `___________________________________________________________`


## 7. Cierre y próximos pasos

### Antes de la próxima sesión
Marca lo que deberías repasar:

- [ ] Scatterplots e interpretación visual
- [ ] Pearson vs Spearman
- [ ] Tipos de variables y métricas adecuadas
- [ ] Confusores y causalidad
- [ ] Segmentación y Simpson’s paradox
- [ ] Redacción de insights

### Mi plan de repaso
- `___________________________________________________________`
- `___________________________________________________________`

### Pregunta que todavía me queda
`___________________________________________________________`


## 8. Información complementaria y recursos

- Documentación oficial de pandas: <https://pandas.pydata.org/docs/>
- Documentación oficial de seaborn: <https://seaborn.pydata.org/>
- Documentación oficial de SciPy stats: <https://docs.scipy.org/doc/scipy/reference/stats.html>
- Temas clave para reforzar:
  - Pearson vs Spearman
  - Point-biserial
  - Correlation ratio (η)
  - Cramér’s V
  - Simpson’s paradox

### Recurso que quiero consultar primero
`___________________________________________________________`


## Cierre

### Mini repaso
1. ¿Qué diferencia hay entre una relación visual y un coeficiente resumido?  
   `___________________________________________________________`

2. ¿Por qué una correlación fuerte no demuestra causalidad?  
   `___________________________________________________________`

3. ¿Qué ejemplo del notebook te ayudó más a entender la idea de confusor o subgrupo?  
   `___________________________________________________________`

### Autoevaluación rápida
- [ ] Entiendo el concepto de correlación
- [ ] Puedo distinguir Pearson y Spearman
- [ ] Puedo escoger una métrica razonable según los tipos de variables
- [ ] Necesito repasar interpretación de gráficos
- [ ] Me siento listo/a para la sesión práctica

### Espacio para dudas
`___________________________________________________________`

### ¿Necesitas ayuda?
- `DATA-CO-LEARNING`: revisa los horarios de apoyo en la guía del estudiante.
- `DA_CONSULTA`: publica dudas de contenido o del proyecto usando el tag correspondiente.
- `SPRINT FOCUS`: sesiones extra para profundizar temas clave del sprint.
- `SESIONES 1:1`: agenda con anticipación si necesitas apoyo individual.


## Siguientes pasos

- **Próxima sesión:** aplicar estas ideas en un notebook práctico con más tiempo para explorar relaciones y documentar hallazgos.
- **Tarea sugerida:** vuelve a ejecutar este notebook y prueba nuevos pares de variables.
- **Recordatorio personal:**  
  `___________________________________________________________`
